# 01 - Basic Context Methods for AI

This notebook demonstrates the fundamental ways to provide context to AI models:
- Basic chat without context
- Static context via system prompts  
- Dynamic context through multi-turn conversation
- Understanding when to use each approach

In [2]:
import os
import json
from openai import AzureOpenAI
from dotenv import load_dotenv
load_dotenv()

# Azure OpenAI Configuration
endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
deployment = os.getenv("AZURE_OPENAI_GPT4O_DEPLOYMENT_NAME")
subscription_key = os.getenv("AZURE_OPENAI_API_KEY")
api_version = os.getenv("AZURE_OPENAI_GPT4O_API_VERSION")

# Initialize Azure OpenAI client
client = AzureOpenAI(
    azure_endpoint=endpoint,
    api_key=subscription_key,
    api_version=api_version,
)

print("✅ Azure OpenAI client initialized successfully!")
print(f"📍 Endpoint: {endpoint}")
print(f"🤖 Deployment: {deployment}")

✅ Azure OpenAI client initialized successfully!
📍 Endpoint: https://foundrynative-resource.openai.azure.com/
🤖 Deployment: gpt-4o


## 🚫 Method 1: Basic Chat (No Context)

Let's start with a simple chat completion without any context to see the AI's limitations.

In [3]:
# Example: Asking for specific information without context
def basic_chat_example():
    """Demonstrate AI limitations without context"""
    messages = [
        {
            "role": "user",
            "content": "What's the current time in Amsterdam and the weather there today?"
        }
    ]
    
    response = client.chat.completions.create(
        model=deployment,
        messages=messages,
        max_tokens=800,
        temperature=0.7
    )
    
    print("🤖 AI Response (No Context):")
    print("=" * 50)
    print(response.choices[0].message.content)
    print("=" * 50)
    print("\n💡 Notice: AI cannot provide real-time data!")
    
    return response

basic_response = basic_chat_example()

🤖 AI Response (No Context):
I can't provide real-time information since I don't have access to live data. However, you can easily check the current time and weather in Amsterdam using a reliable source like [timeanddate.com](https://www.timeanddate.com) or a weather app on your phone. Let me know if I can assist further!

💡 Notice: AI cannot provide real-time data!


## 📋 Method 2: Static Context via System Prompts

System prompts provide static context that shapes the AI's behavior and knowledge base.
Perfect for domain-specific assistants with stable information.

In [4]:
def system_prompt_example():
    """Demonstrate static context through system prompts"""
    messages = [
        {
            "role": "system",
            "content": """You are a Microsoft Azure Solutions Architect with deep expertise in cloud architecture patterns. You have access to the following context about our company's current infrastructure:

- Environment: Production workloads in West Europe and East US regions
- Current Services: 50+ App Services, 10 AKS clusters, 30+ SQL databases
- Monthly Spend: $45,000 USD
- Compliance Requirements: SOC 2, GDPR, HIPAA
- Development Teams: 5 teams using .NET Core, Python, and Node.js
- Pain Points: High database costs, complex networking setup, manual deployments

Provide architectural recommendations based on this context. Always consider cost optimization, security best practices, and our compliance requirements."""
        },
        {
            "role": "user",
            "content": "We're experiencing performance issues with our SQL databases during peak hours. What would you recommend?"
        }
    ]
    
    response = client.chat.completions.create(
        model=deployment,
        messages=messages,
        max_tokens=1000,
        temperature=0.7
    )
    
    print("🤖 AI Response (With System Context):")
    print("=" * 50)
    print(response.choices[0].message.content)
    print("=" * 50)
    print("\n💡 Notice: AI provides specific recommendations based on the company context!")
    
    return response

system_response = system_prompt_example()

🤖 AI Response (With System Context):
To address the performance issues with your SQL databases during peak hours, I recommend the following architectural optimizations:

### 1. **Analyze and Optimize Workloads**
   - **Query Performance Tuning**: Use Azure SQL's Query Performance Insights to identify slow-running queries and optimize them. Encourage developers to use indexes, optimize joins, and avoid expensive operations like SELECT *.
   - **Database Partitioning/Sharding**: If certain tables are excessively large or queries involve high contention, consider partitioning the database or implementing sharding to distribute the workload across multiple databases.

---

### 2. **Scale Appropriately**
   - **Elastic Pools for Cost Efficiency**: If your databases have varying usage patterns, move them to Azure SQL Elastic Pools. Elastic pools allow multiple databases to share compute resources, reducing costs and improving resource utilization.
   - **Vertical Scaling**: During peak hours

## 💬 Method 3: Dynamic Context Through Multi-Turn Conversation

This approach builds context progressively through conversation, allowing for flexible and exploratory interactions.

In [5]:
def multi_turn_example():
    """Demonstrate dynamic context building through conversation"""
    
    # Conversation turns that build context progressively
    conversation = [
        {
            "role": "user",
            "content": "I'm working on a web application for my company."
        },
        {
            "role": "assistant", 
            "content": "I'd be happy to help with your web application! To provide the best guidance, could you tell me more about:\n- What technology stack are you using?\n- What's the main purpose of the application?\n- Are there any specific challenges you're facing?"
        },
        {
            "role": "user",
            "content": "It's an e-commerce platform built with Next.js and Node.js. We're using PostgreSQL for the database and deploying on Azure. We have about 10,000 products and expect 5,000 daily active users."
        },
        {
            "role": "assistant",
            "content": "Great! An e-commerce platform with Next.js, Node.js, and PostgreSQL on Azure is a solid architecture. With 10,000 products and 5,000 DAU, you'll want to focus on performance and scalability. What specific aspect would you like help with? For example:\n- Performance optimization\n- Database design\n- Caching strategies\n- Azure deployment configuration\n- Search functionality\n- Payment integration"
        },
        {
            "role": "user",
            "content": "We're having issues with slow product search. Users complain it takes 3-5 seconds to get results. We're currently using LIKE queries in PostgreSQL."
        }
    ]
    
    response = client.chat.completions.create(
        model=deployment,
        messages=conversation,
        max_tokens=1000,
        temperature=0.7
    )
    
    print("🤖 AI Response (Dynamic Context Built Through Conversation):")
    print("=" * 50)
    print(response.choices[0].message.content)
    print("=" * 50)
    print("\n💡 Notice: AI provides specific solutions based on accumulated context!")
    
    return response, conversation

multi_turn_response, conversation_history = multi_turn_example()

🤖 AI Response (Dynamic Context Built Through Conversation):
The slowness in your product search is likely due to the inefficiency of `LIKE` queries, especially as your product table grows. Here are several strategies to optimize your search functionality:

---

### 1. **Move to Full-Text Search in PostgreSQL**
PostgreSQL has built-in full-text search capabilities, which are significantly faster and more powerful than `LIKE` queries.

#### Steps:
1. **Add a `tsvector` Column for Full-Text Search**:
   Create a `tsvector` column to store pre-processed searchable content (e.g., product names, descriptions).

   ```sql
   ALTER TABLE products
   ADD COLUMN search_vector tsvector;

   UPDATE products
   SET search_vector = to_tsvector('english', name || ' ' || description);
   ```

2. **Create a GIN Index on the `tsvector` Column**:
   Indexing the `tsvector` column improves the performance of search queries.

   ```sql
   CREATE INDEX idx_search_vector ON products USING GIN (search_vector)

## 📊 Compare Context Methods

Let's analyze the differences between these approaches:

In [6]:
def analyze_responses():
    """Compare the different context methods"""
    
    print("📊 CONTEXT METHOD COMPARISON")
    print("=" * 60)
    
    print("\n🚫 BASIC CHAT (No Context):")
    print(f"   📝 Tokens used: {basic_response.usage.total_tokens}")
    print(f"   🎯 Specificity: Low - Generic responses")
    print(f"   ⚡ Setup time: Instant")
    print(f"   🔄 Adaptability: None")
    
    print("\n📋 SYSTEM PROMPT (Static Context):")
    print(f"   📝 Tokens used: {system_response.usage.total_tokens}")
    print(f"   🎯 Specificity: High - Domain-specific")
    print(f"   ⚡ Setup time: One-time configuration")
    print(f"   🔄 Adaptability: Fixed until prompt changes")
    
    print("\n💬 MULTI-TURN (Dynamic Context):")
    print(f"   📝 Tokens used: {multi_turn_response.usage.total_tokens}")
    print(f"   🎯 Specificity: Very High - Tailored to specific needs")
    print(f"   ⚡ Setup time: Builds over conversation")
    print(f"   🔄 Adaptability: Highly flexible")
    
    print("\n📈 TOKEN USAGE ANALYSIS:")
    total_conversation_tokens = sum(len(msg["content"].split()) * 1.3 for msg in conversation_history)  # Rough estimate
    print(f"   🚫 Basic: {basic_response.usage.total_tokens} tokens")
    print(f"   📋 System: {system_response.usage.total_tokens} tokens") 
    print(f"   💬 Multi-turn: {multi_turn_response.usage.total_tokens} tokens")
    print(f"   📊 Conversation history: ~{int(total_conversation_tokens)} tokens")

analyze_responses()

📊 CONTEXT METHOD COMPARISON

🚫 BASIC CHAT (No Context):
   📝 Tokens used: 84
   🎯 Specificity: Low - Generic responses
   ⚡ Setup time: Instant
   🔄 Adaptability: None

📋 SYSTEM PROMPT (Static Context):
   📝 Tokens used: 1099
   🎯 Specificity: High - Domain-specific
   ⚡ Setup time: One-time configuration
   🔄 Adaptability: Fixed until prompt changes

💬 MULTI-TURN (Dynamic Context):
   📝 Tokens used: 1238
   🎯 Specificity: Very High - Tailored to specific needs
   ⚡ Setup time: Builds over conversation
   🔄 Adaptability: Highly flexible

📈 TOKEN USAGE ANALYSIS:
   🚫 Basic: 84 tokens
   📋 System: 1099 tokens
   💬 Multi-turn: 1238 tokens
   📊 Conversation history: ~211 tokens


## 🎯 When to Use Each Method

### 🚫 **Basic Chat** (No Context)
**Best for:**
- Quick, general questions
- Exploratory conversations
- When context isn't needed

**Limitations:**
- No domain expertise
- Cannot access real-time data
- Generic responses

### 📋 **System Prompts** (Static Context)
**Best for:**
- Domain-specific assistants
- Consistent behavior requirements
- Well-defined scenarios
- Product catalogs, company policies

**Limitations:**
- Static information only
- No real-time updates
- Limited flexibility

### 💬 **Multi-Turn Conversation** (Dynamic Context)
**Best for:**
- Complex problem-solving
- Requirements gathering
- Adaptive assistance
- Learning user preferences

**Limitations:**
- Higher token usage
- Context window limits
- Requires multiple API calls

## 🚀 Evolution to Function Calling

While these methods provide context, they still can't access:
- ⏰ Real-time data (current time, weather)
- 🌐 Live APIs (stock prices, news)
- 💾 External databases
- 🔧 System operations

**That's where function calling becomes essential!**

## 🎓 What You've Learned

✅ **Context Fundamentals**: Understanding why AI needs context  
✅ **Static Context**: How system prompts provide domain knowledge  
✅ **Dynamic Context**: Building understanding through conversation  
✅ **Trade-offs**: Token usage vs. specificity vs. flexibility  
✅ **Use Cases**: When to apply each method  

## 🚀 Next Steps

Now that you understand the context provision methods, let's explore how function calling enables real-time capabilities:

- **02_function_calling_legacy.ipynb** - Learn the deprecated format (educational)
- **03_function_calling_modern.ipynb** - Master current best practices  
- **04_multiple_tool_calls.ipynb** - Advanced parallel execution
- **05_assistants_vector_store.ipynb** - Document-based intelligence

**Ready to give your AI real-time superpowers?** Let's continue! 🎯